# 用講的跟 Workflow 對話

這一份示範語音輸入輸出的四個特性：安靜時不上傳、話到了才開跑、被打斷之後只留對方聽到的部分，以及接上真的語音部署。

前三段用假的音訊傳輸驅動，**不需要網路也不需要憑證**，在 Colab 直接跑得動。最後一段才需要 Azure 的語音部署。


## 在 Colab 準備環境


In [ ]:
!pip install -q "git+https://github.com/R300-AI/Agentic-SDK.git"

## 特性一：安靜的時候什麼都不上傳

純靜音和說話的計費相同，而且會被服務辨識成沒有人說過的字。不過濾的話，一個安靜的房間會持續產生假的輸入，agent 會不停打斷自己的回答。

`VoiceTextPerceive` 在送出前先過閘門。說完之後還會多送一小段安靜——因為服務是靠「聽到靜音」判斷一句話結束的，切太乾淨反而永遠等不到轉寫。


In [ ]:
import struct
from agentic_sdk import Workflow
from agentic_sdk.audio import FakeAudioInput
from agentic_sdk.modules import DirectAnswerAction, PassThroughRetrieve, VoiceTextPerceive

def pcm(*samples):
    return struct.pack(f"<{len(samples)}h", *samples)

def silence(frames=1600):
    return pcm(*([0] * frames))

def speech(frames=1600, level=8000):
    return pcm(*([level, -level] * (frames // 2)))

audio = FakeAudioInput()
perceive = VoiceTextPerceive(transport=audio)

for chunk in [silence(), silence(), speech(), speech(), silence(), silence(), silence()]:
    perceive.hear(chunk)

print("送出去的片段數：", len(audio.sent))
print("其中有聲音的：", sum(1 for chunk in audio.sent if chunk != silence()))

## 特性二：話到了才開跑

一般的回合是呼叫端先說這一輪要處理什麼。語音不是：話什麼時候來，取決於人什麼時候想講。所以模組會先把聽到的收著，`Workflow.run()` 不必再被告知一次。

收著的內容用掉就沒了，不會殘留到下一輪。


In [ ]:
workflow = Workflow(
    workflow_name="語音問答",
    perceive=perceive,
    retrieve=PassThroughRetrieve(),
    action=DirectAnswerAction(),
)
audio.transcribe("保固多久？")
print("開跑前拿到的話：", perceive.pending_input())
result = workflow.run()
print("回覆：", result.final_message)
print("再問一次時已經沒有殘留：", repr(perceive.pending_input()))

## 特性三：被打斷之後，只留對方聽到的那一段

語音落後生成。答案早就寫完了，聲音還在播——所以被打斷的那一輪，有一段是寫出來但沒有人聽過的。

把那一段留在上下文裡，下一輪 agent 就會引用一句沒人聽過的話。`heard_portion` 用播放時長把它裁掉；沒有人回報時長時就整段保留，因為「不知道」不等於「沒聽到」。


In [ ]:
from agentic_sdk.audio.speech_rate import heard_portion

written = "保固期是十二個月，延長保固可以再加兩年，另外配件另計"
print("整段寫好的：", written)
print("只播了兩秒：", heard_portion(written, 2.0))
print("沒有人回報時間時：", heard_portion(written, None))

## 打斷是完整的一輪，不是錯誤

偵測靠的是語音活動，不是聽懂了什麼——開口約 600 毫秒就測得到，轉寫要將近四秒，等字就等於繼續講在別人身上。

注意 `aborted` 是 `False`：那個旗標是流程自我中止（跳轉上限、逾時）才用的，會被當成錯誤顯示給使用者。有人故意插話不是錯誤。


In [ ]:
from agentic_sdk.audio import FakeAudioOutput
from agentic_sdk.core import ContextEntry, ContextEntryType, ModuleOutput
from agentic_sdk.core.cancellation import CancellationToken

class SlowAnswer:
    """一個講很久的 action，讓我們有時間插話。"""
    name = "action"

    def __init__(self, microphone, speaker):
        self._microphone = microphone
        self._speaker = speaker

    def __call__(self, state):
        text = "保固期是十二個月，延長保固可以再加兩年，另外配件另計"
        for index, piece in enumerate(self._speaker.speak(text)):
            if index == 1:
                self._microphone.start_speaking()   # 使用者在這裡開口
            state.report_spoken_progress(heard_portion(text, 2.0))
            if state.should_stop():
                break
        state.cancel.raise_if_cancelled()
        return ModuleOutput(next_module=None, payload={"latest_final_message": text})

microphone = FakeAudioInput()
listening = VoiceTextPerceive(transport=microphone)
talking = Workflow(
    workflow_name="會被打斷的語音問答",
    perceive=listening,
    action=SlowAnswer(microphone, FakeAudioOutput()),
)
microphone.transcribe("保固多久？")
interrupted = talking.run(cancel=CancellationToken())

print("被打斷了嗎：", interrupted.interrupted)
print("這是錯誤嗎：", interrupted.aborted)
print("對方實際聽到：", interrupted.interrupt_payload["heard"])
print("記憶裡留下的：", talking.memory.turns[-1].content)

## 接上真的語音部署

把假的傳輸換成三件式端點參數就是真的了，其餘程式碼不動。

**SDK 不擷取麥克風、也不播放聲音。** 進來的音訊由你餵給 `hear()`；出去的音訊要包一層在 `speak()` 裡交給音訊裝置再 `yield`——先播放再 `yield`，插話時放棄串流才會同時停掉播放與合成。

可執行的完整範例在 repo 的 `examples/voice/desktop_voice_agent.py`，用一個 WAV 當麥克風，沒有音效裝置也跑得動。


In [ ]:
from agentic_sdk.modules import VoiceAnswerAction, VoiceTextPerceive

perceive = VoiceTextPerceive(
    api_key="<TRANSCRIBE_API_KEY>",
    base_url="<TRANSCRIBE_BASE_URL>",
    model="<TRANSCRIBE_DEPLOYMENT>",
)
action = VoiceAnswerAction(
    api_key="<CHAT_API_KEY>",
    base_url="<CHAT_BASE_URL>",
    model="<CHAT_DEPLOYMENT>",
    speech_api_key="<TTS_API_KEY>",
    speech_base_url="<TTS_BASE_URL>",
    speech_model="<TTS_DEPLOYMENT>",
)